# Big Data Analytics: Project 1
**From 2,000+ social-media records to a marketing decision**
Student: Salman · University of Management and Technology (UMT)

**Business question:** A clothing brand posts on social media. Which platforms and kinds of posts get the most engagement, and what should the marketing team test next?

## Step 1–2: Business question and course-topic mapping
| Course topic | Where it appears in this project |
|---|---|
| What is Big Data? | Step 8: comparing 2k rows with a much larger workload |
| Why Big Data? | Step 8: runtime, memory and file size as data grows |
| Data sources | Social-media posts with likes, shares, comments, hashtags, text |
| Types of data | Text, dates, categories, counts (see data dictionary in Step 5) |
| EDA and data objects | Steps 5–6: each row is one post |
| Plots and statistics | Step 6–7: counts, mean/median, bar chart, histogram, scatter |
| Analytics techniques | Step 9: classification, clustering, regression |

## Step 3: Dataset source and provenance
- **Source:** GitHub repository `mrudukp/social-media-sentiment-analysis`, file `social_media_sentiment_raw.csv` (author: Mrudula KP). 
- **Size:** 2,260 rows × 13 columns, five platforms, 12 countries, Jan 2023 – Dec 2024.
- **License:** *Check the repository's license/README before submitting and write it here.* No license file was found in the repository files I inspected.
- **Caveat:** This data is **not verified** as coming from Pakistani clothing brands. It is used to practise the workflow, so findings are about *this dataset only*, not about Pakistan or any specific brand.
- A raw backup copy is kept: the original CSV is never edited, all cleaning happens on a copy.

## Step 4: Load the data (Colab-ready)

In [ ]:
import pandas as pd, numpy as np, time
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/mrudukp/social-media-sentiment-analysis/main/social_media_sentiment_raw.csv"
try:
    raw = pd.read_csv(URL)
except Exception:
    raw = pd.read_csv("social_media_sentiment_raw.csv")   # fallback: upload the CSV to Colab

df = raw.copy()          # raw stays untouched as our backup
print(df.shape)
display(df.head())
df.info()

## Step 5: Data objects, attributes and quality checks
**Observational unit:** one row = **one social-media post**.

### Data dictionary
| Column | Type | Meaning |
|---|---|---|
| post_id | identifier | Unique post ID |
| timestamp | date/time | When the post was made |
| platform | categorical | Twitter, Instagram, Facebook, TikTok, LinkedIn |
| user | text/identifier | Account handle |
| followers_count | numerical | Followers of the account |
| verified | categorical | Yes / No verified account |
| text | text | Post content |
| hashtags | text | Hashtags used (blank = none or not recorded) |
| country | categorical | Country of the account |
| likes | numerical | Number of likes |
| retweets_shares | numerical | Number of shares/retweets |
| comments | numerical | Number of comments |
| sentiment | categorical | Positive / Neutral / Negative label |

⚠️ There is **no reach or impressions** column, so we cannot compute a true engagement rate by reach.

In [ ]:
print("Rows, columns:", df.shape)
display(df.dtypes)
display(df.isna().sum())
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate post_id values:", df['post_id'].duplicated().sum())
print("Negative likes:", (df['likes'] < 0).sum())
print(df['platform'].value_counts(dropna=False))
display(df['country'].value_counts(dropna=False).head(30))
display(df.describe(include='all').T)

**Problems found (and how we handle them):**
1. **Inconsistent text** in `platform` ("TWITTER" vs "Twitter") and `country` ("usa  " vs "USA") → standardise case and strip spaces.
2. **Duplicate rows** → drop exact duplicates.
3. **`followers_count` stored as text**, some with commas → convert to numbers.
4. **Negative likes** are impossible → set to missing (not to zero, because we do not know the true value).
5. **Missing values** (`hashtags`, `country`, `followers_count`, `verified`, `comments`, `sentiment`) → *not deleted blindly.* A blank hashtag likely means "no hashtag"; a blank metric may mean "not collected". We keep the rows and let each analysis skip the missing cells.

In [ ]:
df = raw.drop_duplicates().copy()

df['platform'] = df['platform'].str.strip().str.title()
df['country'] = df['country'].str.strip().str.upper()
df['country'] = df['country'].where(df['country'].isin(['USA','UK','UAE']), df['country'].str.title())

df['followers_count'] = pd.to_numeric(df['followers_count'].astype(str).str.replace(',', ''), errors='coerce')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.loc[df['likes'] < 0, 'likes'] = np.nan

df['hashtag_count'] = df['hashtags'].fillna('').apply(lambda s: len(s.split()) if s else 0)
df['day_of_week'] = df['timestamp'].dt.day_name()

print("Before:", raw.shape, " After:", df.shape)
print(df['platform'].value_counts())
print("Countries:", df['country'].nunique())

## Step 6: Exploratory data analysis (EDA)

In [ ]:
metrics = ['likes', 'retweets_shares', 'comments']
by_platform = df.groupby('platform')[metrics].agg(['count', 'mean', 'median']).round(2)
display(by_platform)
display(df[metrics].describe().round(2))
display(df[metrics].corr().round(2))

In [ ]:
# Chart 1: median likes per platform (with number of posts shown)
stats = df.groupby('platform')['likes'].agg(['median', 'count']).sort_values('median', ascending=False)
ax = stats['median'].plot(kind='bar', color='steelblue', figsize=(7,4))
for i, (m, n) in enumerate(zip(stats['median'], stats['count'])):
    ax.text(i, m + 0.5, f"n={n}", ha='center')
plt.title('Median likes per post, by platform')
plt.xlabel('Platform'); plt.ylabel('Median likes'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2: distribution of likes
df['likes'].plot(kind='hist', bins=30, color='darkorange', figsize=(7,4))
plt.axvline(df['likes'].mean(), color='k', linestyle='--', label=f"mean = {df['likes'].mean():.1f}")
plt.axvline(df['likes'].median(), color='r', linestyle=':', label=f"median = {df['likes'].median():.1f}")
plt.title('Distribution of likes per post'); plt.xlabel('Likes'); plt.ylabel('Number of posts')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Chart 3: relationship between likes and comments
plt.figure(figsize=(6,4))
plt.scatter(df['likes'], df['comments'], alpha=0.3, s=12)
plt.title('Likes vs comments'); plt.xlabel('Likes'); plt.ylabel('Comments')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 4: median likes by sentiment
s = df.groupby('sentiment')['likes'].agg(['median','count'])
display(s)
s['median'].plot(kind='bar', color='seagreen', figsize=(6,4))
plt.title('Median likes by sentiment'); plt.xlabel('Sentiment'); plt.ylabel('Median likes'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## Step 7: Business interpretation

**Observation 1: Platform matters.** TikTok posts have the highest median likes (44) and Instagram is next (36), while LinkedIn is lowest (20). Evidence: Chart 1 and the platform table. Group sizes are different (LinkedIn has only ~119 posts versus 734 for Twitter), so the ranking is clear but LinkedIn's figure is based on fewer posts.

**Observation 2: Positive posts get more likes.** Positive posts have a median of about 39 likes versus 32 for negative and 28 for neutral. Evidence: Chart 4. Likes and comments also move together (correlation ≈ 0.66, Chart 3), and likes are only mildly skewed (mean ≈ median, Chart 2), so a few viral posts are *not* dominating this dataset.

**Recommendation:** The brand should test publishing more short-form, positive-toned content on TikTok and Instagram and compare results with one consistent metric (for example median likes per post) over a few weeks, before moving budget away from other platforms.

**Limitations:** This is observational data. We cannot prove that platform or sentiment *causes* higher engagement. There is no reach/impressions column, so we cannot calculate true engagement rate or account for audience size. The data is not from a Pakistani clothing brand, and organic data alone cannot measure paid-ad performance (e.g., ROAS).

## Step 8: Scaling experiment: when does a scalable tool become useful?
We repeat the same simple analysis on bigger copies of the data. **These are repeated copies, a workload test and not new evidence.** Results depend on the machine used.

In [ ]:
import os, tempfile
results = []
for factor in [1, 10, 100, 1000]:
    big = pd.concat([df] * factor, ignore_index=True)
    path = os.path.join(tempfile.gettempdir(), f"scale_{factor}.csv")
    big.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / 1e6

    t0 = time.time(); loaded = pd.read_csv(path); t_load = time.time() - t0
    t0 = time.time(); _ = loaded.groupby('platform')[['likes','comments']].agg(['mean','median']); t_an = time.time() - t0
    mem_mb = loaded.memory_usage(deep=True).sum() / 1e6

    results.append({'rows': len(loaded), 'file_MB': round(size_mb, 1), 'load_s': round(t_load, 2),
                    'analysis_s': round(t_an, 3), 'memory_MB': round(mem_mb, 1)})
    del big, loaded
    os.remove(path)

scale = pd.DataFrame(results)
display(scale)

In [ ]:
scale.plot(x='rows', y=['load_s', 'analysis_s'], marker='o', figsize=(6,4), logx=True)
plt.title('Time vs number of rows'); plt.ylabel('Seconds'); plt.tight_layout(); plt.show()

**Conclusion (write your own numbers from the table above):** Runtime, file size and memory grow roughly in proportion to the number of rows. Pandas copes well at this size and may even handle millions of rows on a good computer, so there is **no fixed row count where it "fails"**. A scalable tool such as Spark becomes worth investigating when the data no longer fits in memory, when processing is too slow for the business to wait, or when the data must be stored across many machines. For a ~2,000-row file it is not needed.

## Step 9: Connecting to analytics techniques
| Technique | How it could help the marketer | What it would need |
|---|---|---|
| **Classification** | Predict whether a post will have high or low engagement | A defensible label, enough posts in both classes, features that are known *before* posting |
| **Clustering** | Group posts with similar behaviour to find content styles | Scaled numerical features |
| **Regression** | Predict the number of likes | A numeric target and informative predictors |

**Avoid data leakage:** do not create a label from `likes` and then use `likes` as a feature. Always split into training and test sets.

Short demo: predict "high engagement" (likes above the median) using only features known before posting.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

d = df.dropna(subset=['likes']).copy()
d['high_engagement'] = (d['likes'] > d['likes'].median()).astype(int)

features = ['platform', 'verified', 'followers_count', 'hashtag_count']   # NO likes, shares or comments (leakage)
X, y = d[features], d['high_engagement']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pre = ColumnTransformer([
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), ['platform', 'verified']),
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), ['followers_count', 'hashtag_count'])])
model = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))]).fit(X_train, y_train)

print("Test accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 3))
print("Baseline (always predict majority class):", round(max(y_test.mean(), 1 - y_test.mean()), 3))

**Exit question:** Which technique helps my marketing question? *Classification* (or regression) fits best, since the question is which posts get more engagement. It needs a defensible label, features known before posting, and a proper train/test split. Clustering would help only if we wanted to discover content styles without a target.

## Step 10: Submission checklist
- [x] Dataset source, provenance note and data dictionary (Steps 3, 5)
- [x] Record count, column types, missing-value and duplicate checks (Step 5)
- [x] Two summary statistics and two charts (Step 6: four charts included)
- [x] Two evidence-based observations and one recommendation (Step 7)
- [x] One limitation and a reason a scalable tool might help (Steps 7–8)
- [x] Analytics concept (Step 9)

> **Closing line:** What does the data tell us, what business problem does it reveal, and what should the business do next?
> The data suggests TikTok and Instagram posts, and positive-toned posts, earn more likes; the business should run a controlled test of more content there and measure it with a consistent metric.